**[Source]** Jisoo Project (`지수프젝/2_전처리_v3b.ipynb`, 원본 SHA256 앞 16자리 `fc3f57f09ca431d1`)
**[Status]** FIXED
**[Role]** 전처리 + document_id 기반 Train/Validation/Test 분할 + 품질 플래그 (preprocessed_v3b 생성)
**[Modification]** 원본 코드 수정 금지. 이 파일은 원본의 복사본이며 **맨 위에 이 안내 셀 1개만 추가**했다(코드·출력 셀은 원본과 동일, config/fixed_manifest.json의 code_sha256으로 확인).
**[Rerun]** 재실행하지 않는다. 이미 실행된 결과(`data/preprocessed_final`)를 05번 노트북에서 독립 검증한 뒤 사용한다.

# 전처리 노트북: NIKL 맞춤법 교정 말뭉치 → 학습용 문서 단위 분할 데이터

이 노트북은 **모델 학습 직전** 단계입니다. 모델 학습 코드는 없습니다. `EDA.ipynb`에서 확인한 데이터 특성(빈 행, `hate-speech` 자리표시자, 같은 form에 여러 교정, 문장부호·토큰 변화 등)을 반영해 아래 순서로 처리합니다.

| 단계 | 하는 일 | 왜 이 순서인가 |
|---|---|---|
| 1 | 원본 JSON을 읽기 전용 스트리밍으로 파싱해 발화 단위 표로 펼침 | 이스케이프(`\"`, `\r\n`, `\uXXXX`)를 JSON 파서가 처리하게 해서 문자열 깨짐을 막음 |
| 2 | NFC 정규화 + 앞뒤 공백 제거로 `input`, `target` 생성 (원문은 별도 컬럼 보존) | 다른 텍스트 변형은 하지 않음 |
| 3 | 빈 행, `hate-speech` 포함 행, 이모지만 있는 행, 정답이 깨진 행(정답이 지나치게 길어진 행·비식별화 토큰 변경)을 **제외 대상으로 표시** (삭제 아님, 사유 저장) | 행 자체의 내용만으로 결정되므로 분할 전에 해도 누수가 없음 |
| 4 | **문서 단위** 90:5:5 분할 (메타데이터와 입력 문장만 사용) | 정답(target) 통계를 쓰지 않으므로 분할이 품질 판단에 오염되지 않음 |
| 5 | 품질 판단용 통계·규칙을 **train에서만** 생성 | validation/test 정보가 train 판단에 들어가지 않게 함 |
| 6 | 같은 규칙을 train/validation/test에 동일하게 적용해 **품질 플래그** 부여 (삭제 아님) | 의심 사례를 정답 오류로 단정하지 않음 |
| 7 | 분할별 제외 수·플래그 수·사용 가능 행 수 요약 | 결과를 표와 JSON으로 남김 |
| 8 | 새 폴더에 저장하고 다시 읽어서 검증 | 기존 파일을 덮어쓰지 않음 |

**실행 방법**: 프로젝트 루트(`data/` 폴더가 있는 곳)에서 이 노트북을 열고 위에서 아래로 순서대로 실행합니다. `pandas`, `numpy`만 필요합니다. 결과는 `data/preprocessed_v1/`(새 폴더)에 저장됩니다.

**지키는 원칙**
- 원본 JSON은 읽기 전용으로만 열고, 시작과 끝의 SHA-256을 비교합니다.
- `corrected_form`은 어떤 모델 출력으로도 덮어쓰지 않습니다. 외부 모델(ET5 등)은 쓰지 않습니다.
- 플래그는 "의심 사례 표시"이며 정답 오류 확정이 아닙니다.
- 저장 폴더가 이미 있고 비어 있지 않으면 **중단**합니다(덮어쓰기 방지).

In [1]:
# 0. 설정 - 경로, 시드, 규칙 임계값. 실행 결과에 영향을 주는 값은 모두 여기에 모아 두고 결과 JSON에도 저장합니다.
import os, sys, re, json, hashlib, time, unicodedata, platform, datetime, difflib
from pathlib import Path
from collections import Counter, defaultdict

import warnings
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)   # 한글 표 열 정렬(전각 폭 반영)
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")   # 역참조가 있는 정규식의 경고 무시

# 노트북이 있는 폴더(프로젝트 루트)에서 실행한다고 가정합니다. 다른 곳이면 환경변수로 바꿀 수 있습니다.
BASE_DIR = Path(os.environ.get("NIKL_PROJECT_DIR", ".")).resolve()
DATA_PATH = Path(os.environ.get("NIKL_DATA_PATH", str(BASE_DIR / "data" / "MXEC2202210100.json"))).resolve()
OUT_DIR = Path(os.environ.get("NIKL_PREPROCESS_OUT_DIR", str(BASE_DIR / "data" / "preprocessed_v3b"))).resolve()
MAX_DOCS = int(os.environ["NIKL_MAX_DOCS"]) if os.environ.get("NIKL_MAX_DOCS") else None   # 빠른 점검용(기본: 전체)

SEED = 42
INCLUDE_RAW_COLUMNS_IN_SPLIT_FILES = True   # False로 바꾸면 split 파일에서 원문 3개 컬럼을 빼서 파일이 작아집니다(train 약 1.1GB → 더 작게)
SPLITS = ["train", "validation", "test"]
SPLIT_RATIOS = {"train": 0.90, "validation": 0.05, "test": 0.05}

# --- 제외 규칙 (행 내용만으로 결정) ---
HATE_RE = re.compile(r"hate[\s_\-]*speech", re.IGNORECASE)   # 'hate-speech' 자리표시자 (표기 변형 포함)

# --- 임계값: EDA에서 정한 사전 정의 값입니다 (train 라벨 통계로 학습한 값이 아님). LEN_* 값 중 상한(LEN_RATIO_HI)은 제외 기준으로, 하한(LEN_RATIO_LO)은 정보용 플래그로 쓰입니다 ---
LEN_RATIO_LO, LEN_RATIO_HI, LEN_ABS_MIN = 0.7, 1.5, 3   # 공백·문장부호 제외 글자 수의 target/input 비율
EDIT_RATIO_HI = 0.5                                     # 철자·단어 수준 변경 중 편집량(1 - difflib 유사도)
PUNCT_ADD_MIN = 3                                       # 문장부호 증가량
PUNCT_RUN_MIN = 4                                       # 같은 부호가 이만큼 연속으로 새로 생기면 과다
CONFLICT_MIN_LEN = 5                                    # '긴 입력'의 기준(공백 제외 글자 수)
RULE_MIN_SUPPORT, RULE_MIN_RATE = 20, 0.9               # train에서 학습하는 단어 교정 규칙의 최소 근거

# --- 문서 묶음(같은 긴 문장을 공유하는 문서는 같은 split에 배정) ---
GROUP_NEAR_DUP_DOCS = True
DUP_MIN_LEN, DUP_MIN_HANGUL = 15, 5

PUNCT_CHARS = set(".,!?~…·\"'()[]{}:;-/\\")
_PUNCT_INNER = re.escape("".join(sorted(PUNCT_CHARS)))
PUNCT_RE = "[" + _PUNCT_INNER + "]"
PUNCT_STRIP_RE = r"[\s" + _PUNCT_INNER + "]"
PUNCT_TABLE = str.maketrans("", "", "".join(PUNCT_CHARS))

# --- '이모지만 있는 행' 판정 (글자·숫자가 하나도 없고 이모지/그림 기호만 있는 행) ---
# 이모지 본체: 이모티콘·그림문자 블록, 기타 기호(Misc Symbols)·딩뱃(Dingbats), 기타 기호 및 화살표, 기술 기호(⌚⏰ 등)
EMOJI_BASE = "[\U0001F000-\U0001FAFF\u2600-\u27BF\u2B00-\u2BFF\u2300-\u23FF\u203C\u2049\u2122\u2139\u24C2\u3030\u303D\u3297\u3299\u00A9\u00AE]"
EMOJI_MOD = "[\uFE0F\u200D\u20E3]"   # 변형 선택자, 이어붙임(ZWJ), 키캡
# 공백·문장부호는 함께 있어도 허용하되, 이모지 본체가 1개 이상 있어야 함. 한글 자모(ㅋㅋ, ㅠㅠ)나 ^^ 같은 글자 이모티콘은 여기에 해당하지 않음
EMOJI_ONLY_RE = re.compile(r"(?:%s|%s|%s)*%s(?:%s|%s|%s)*" % (EMOJI_MOD, EMOJI_BASE, PUNCT_STRIP_RE, EMOJI_BASE, EMOJI_MOD, EMOJI_BASE, PUNCT_STRIP_RE))

DEID_NAMES = ["address", "affiliation", "num", "account", "others", "telNum"]   # EDA에서 확인한 &...& 토큰 종류 (name1, name2 ... 는 name\d+)
# name1, name2 ... 는 앞뒤에 글자가 붙어 있어도(예: aename6, name3name4) 토큰으로 봅니다. 나머지는 단어 경계가 있을 때만 토큰으로 봅니다.
DEID_RE = re.compile(r"name\d+|(?<![A-Za-z])(?:%s)(?![A-Za-z])" % "|".join(sorted(DEID_NAMES, key=len, reverse=True)), re.IGNORECASE)

# --- 저장 폴더 보호 ---
assert DATA_PATH.exists(), f"원본 JSON을 찾을 수 없습니다: {DATA_PATH}"
assert OUT_DIR != DATA_PATH.parent, "저장 폴더가 원본 데이터 폴더와 같습니다. 새 폴더를 지정하세요."
if OUT_DIR.exists() and any(OUT_DIR.iterdir()):
    raise FileExistsError(f"저장 폴더가 이미 있고 비어 있지 않습니다: {OUT_DIR}\n기존 결과를 덮어쓰지 않기 위해 중단합니다. OUT_DIR 이름을 바꿔 다시 실행하세요.")

ENV_INFO = {"python": sys.version.split()[0], "platform": platform.platform(), "pandas": pd.__version__, "numpy": np.__version__}
print("데이터:", DATA_PATH)
print("저장 폴더(새 폴더):", OUT_DIR)
print("환경:", ENV_INFO)
if MAX_DOCS:
    print(f"※ 점검용 실행: 문서 {MAX_DOCS}개만 사용합니다.")

데이터: C:\Users\KDS-18\Desktop\pro4_team3\data\MXEC2202210100.json
저장 폴더(새 폴더): C:\Users\KDS-18\Desktop\pro4_team3\data\preprocessed_v3b
환경: {'python': '3.11.9', 'platform': 'Windows-10-10.0.26200-SP0', 'pandas': '2.2.2', 'numpy': '1.26.4'}


## 1. 원본 JSON 안전 파싱 (읽기 전용, 스트리밍)

517MB JSON을 통째로 올리지 않고, **표준 `json` 파서(`raw_decode`)로 문서 하나씩** 읽습니다. 정규식으로 줄에서 값을 뽑지 않으므로 따옴표·줄바꿈·유니코드 이스케이프가 JSON 규칙대로 복원됩니다. 파일은 `"r"` 모드로만 열고 시작·끝 SHA-256으로 변경 여부를 확인합니다.

In [2]:
# 1-a. 원본 무결성(시작) + 스트리밍 파서
def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

t0 = time.time()
SRC_SHA_START = sha256_of(DATA_PATH)
SRC_SIZE = DATA_PATH.stat().st_size
print(f"원본 SHA-256(시작): {SRC_SHA_START}  ({SRC_SIZE:,} bytes, {time.time()-t0:.0f}s)")

_WS = " \t\r\n"
_DEC = json.JSONDecoder()

class _JsonStream:
    """파일을 청크 단위로 읽으면서 JSON 값을 하나씩 디코딩합니다 (문자열 이스케이프는 json 모듈이 처리)."""
    def __init__(self, fh, chunk=1 << 23):
        self.fh, self.chunk, self.buf, self.pos, self.eof = fh, chunk, "", 0, False

    def _fill(self):
        data = self.fh.read(self.chunk)
        if not data:
            self.eof = True
            return False
        self.buf = self.buf[self.pos:] + data
        self.pos = 0
        return True

    def peek(self):
        while True:
            n = len(self.buf)
            while self.pos < n and self.buf[self.pos] in _WS:
                self.pos += 1
            if self.pos < n:
                return self.buf[self.pos]
            if not self._fill():
                return ""

    def expect(self, ch):
        got = self.peek()
        if got != ch:
            raise ValueError(f"JSON 구조가 예상과 다릅니다: '{ch}' 대신 '{got}'")
        self.pos += 1

    def value(self):
        self.peek()
        while True:
            try:
                obj, end = _DEC.raw_decode(self.buf, self.pos)
            except json.JSONDecodeError:
                if self.eof or not self._fill():
                    raise
                continue
            if end == len(self.buf) and not self.eof and self._fill():   # 값이 청크 끝에서 잘렸을 가능성
                continue
            self.pos = end
            return obj

def iter_documents(path, header_out=None):
    """최상위 {"id":..., "metadata":..., "document":[...]} 에서 document 배열의 원소를 하나씩 내보냅니다."""
    with open(path, "r", encoding="utf-8-sig", newline="") as fh:       # 읽기 전용
        s = _JsonStream(fh)
        s.expect("{")
        while s.peek() != "}":
            key = s.value()
            s.expect(":")
            if key == "document":
                s.expect("[")
                while True:
                    c = s.peek()
                    if c == "]":
                        s.pos += 1
                        break
                    if c == ",":
                        s.pos += 1
                        continue
                    yield s.value()
            else:
                val = s.value()
                if header_out is not None:
                    header_out[key] = val
            if s.peek() == ",":
                s.pos += 1

def flatten_document(doc):
    meta = doc.get("metadata") or {}
    setting = meta.get("setting") or {}
    speakers = {str(sp.get("id")): sp for sp in (meta.get("speaker") or [])}
    utts = doc.get("utterance") or []
    out = []
    for i, u in enumerate(utts):
        spk_id = u.get("speaker_id")
        sp = speakers.get(str(spk_id), {})
        out.append((
            doc.get("id"), u.get("id"), None if spk_id is None else str(spk_id), i, len(utts),
            meta.get("publisher"), meta.get("topic"), setting.get("relation"), meta.get("date"),
            sp.get("sex"), sp.get("age"),
            u.get("original_form"), u.get("form"), u.get("corrected_form"),
            "form" not in u, "corrected_form" not in u,
        ))
    return out

RAW_COLS = ["document_id", "utterance_id", "speaker_id", "utterance_index", "doc_n_utterances",
            "publisher", "topic", "relation", "date", "speaker_sex", "speaker_age",
            "original_form", "form_raw", "corrected_form_raw", "form_missing", "corrected_form_missing"]

원본 SHA-256(시작): 592e81cf9c41f6ee7e30dc660cb8d3f1c282275deca7eca25d64ab1a9acdec39  (517,399,411 bytes, 0s)


In [3]:
# 1-b. 발화 단위 표로 펼치기
t0 = time.time()
header, rows, n_docs = {}, [], 0
for doc in iter_documents(DATA_PATH, header):
    rows.extend(flatten_document(doc))
    n_docs += 1
    if n_docs % 5000 == 0:
        print(f"  ... 문서 {n_docs:,}개 / 발화 {len(rows):,}개 ({time.time()-t0:.0f}s)")
    if MAX_DOCS and n_docs >= MAX_DOCS:
        break
df = pd.DataFrame(rows, columns=RAW_COLS)
del rows

for c in ["original_form", "form_raw", "corrected_form_raw"]:
    df[c] = df[c].fillna("").astype(str)                              # 필드가 없으면 빈 문자열로 두고 *_missing 컬럼에 기록

print(f"파싱 완료 {time.time()-t0:.0f}s | 문서 {df['document_id'].nunique():,} | 발화 {len(df):,}")
print("파일 헤더 키:", list(header.keys()))
assert not df["utterance_id"].duplicated().any(), "발화 ID가 중복됩니다."
assert df["utterance_id"].notna().all() and df["document_id"].notna().all()
print("form 필드 자체가 없는 발화:", int(df["form_missing"].sum()), "/ corrected_form 필드 자체가 없는 발화:", int(df["corrected_form_missing"].sum()))
if not MAX_DOCS:
    print("EDA 기준값과 비교 -> 발화 1,129,363 / 문서 21,335:", len(df) == 1129363 and df["document_id"].nunique() == 21335)
display_cols = ["document_id", "utterance_id", "speaker_id", "publisher", "form_raw", "corrected_form_raw"]
df[display_cols].head(5)

  ... 문서 5,000개 / 발화 196,051개 (1s)
  ... 문서 10,000개 / 발화 505,708개 (2s)
  ... 문서 15,000개 / 발화 690,027개 (3s)
  ... 문서 20,000개 / 발화 861,010개 (4s)
파싱 완료 6s | 문서 21,335 | 발화 1,129,363
파일 헤더 키: ['id', 'metadata']
form 필드 자체가 없는 발화: 0 / corrected_form 필드 자체가 없는 발화: 0
EDA 기준값과 비교 -> 발화 1,129,363 / 문서 21,335: True


,document_id,utterance_id,speaker_id,publisher,form_raw,corrected_form_raw
0,MDRW2100000002.1,MDRW2100000002.1.1,2,카카오톡,하이하이,하이하이.
1,MDRW2100000002.1,MDRW2100000002.1.2,1,카카오톡,반가워욬ㅌㅋㅋ,반가워요. ㅋㅌㅋㅋ
2,MDRW2100000002.1,MDRW2100000002.1.3,1,카카오톡,name2님 제 이상형은,name2 님 제 이상형은
3,MDRW2100000002.1,MDRW2100000002.1.4,1,카카오톡,코가 예쁘면 일단 외관 통관데 name2님은 어때여,코가 예쁘면 일단 외관 통관데 name2 님은 어때요?
4,MDRW2100000002.1,MDRW2100000002.1.5,2,카카오톡,오 저는 무조건 무쌍 존잘이여,"오, 저는 무조건 무쌍 존잘이요."


## 2. 텍스트 정규화 (NFC + 앞뒤 공백 제거만)

`form` → `input`, `corrected_form` → `target`에는 **NFC 정규화와 앞뒤 공백 제거만** 적용합니다. 문장 안쪽의 줄바꿈·공백·문장부호, 대소문자 등은 바꾸지 않습니다. 원문(`original_form`, `form_raw`, `corrected_form_raw`)은 별도 컬럼으로 남깁니다.

아래 표는 이 정규화가 실제로 몇 건을 바꿨는지 보여 줍니다. 이후 단계의 "변경 유형"과 길이 값도 여기서 함께 계산하지만, 모두 행 하나의 내용만으로 정해지는 값입니다(다른 행이나 split의 통계를 쓰지 않음).

In [4]:
def nfc_strip(s: pd.Series) -> pd.Series:
    return s.str.normalize("NFC").str.strip()

df["input"] = nfc_strip(df["form_raw"])
df["target"] = nfc_strip(df["corrected_form_raw"])

nfc_only_in = (df["form_raw"].str.normalize("NFC") != df["form_raw"]).sum()
nfc_only_tg = (df["corrected_form_raw"].str.normalize("NFC") != df["corrected_form_raw"]).sum()
print(pd.DataFrame({
    "form → input": [int((df["input"] != df["form_raw"]).sum()), int(nfc_only_in)],
    "corrected_form → target": [int((df["target"] != df["corrected_form_raw"]).sum()), int(nfc_only_tg)],
}, index=["정규화로 값이 바뀐 행 (공백 제거 또는 NFC)", "그중 NFC 때문에 바뀐 행"]).to_string())

# 행 단위 파생값 (다른 행의 통계를 쓰지 않음)
df["input_len"] = df["input"].str.len()
df["target_len"] = df["target"].str.len()
valid = df["input"].ne("") & df["target"].ne("")

ns_in = df["input"].str.replace(r"\s+", "", regex=True)
ns_tg = df["target"].str.replace(r"\s+", "", regex=True)
same = df["input"].eq(df["target"])
same_nospace = ns_in.eq(ns_tg)
same_nopunct = ns_in.str.translate(PUNCT_TABLE).eq(ns_tg.str.translate(PUNCT_TABLE))
CHANGE_LABELS = ["unchanged", "spacing_only", "punct_only", "spelling_or_word"]
df["change_type"] = pd.Categorical.from_codes(np.select([same, same_nospace, same_nopunct], [0, 1, 2], default=3), CHANGE_LABELS)
df.loc[~valid, "change_type"] = np.nan                                  # 빈 행은 변경 유형을 정의하지 않음
df["content_len_in"] = ns_in.str.translate(PUNCT_TABLE).str.len()      # 공백·문장부호 제외 글자 수
df["content_len_tg"] = ns_tg.str.translate(PUNCT_TABLE).str.len()
del ns_in, ns_tg, same, same_nospace, same_nopunct

print("\n[변경 유형 - 빈 행 제외, 전체 데이터]")
print(df.loc[valid, "change_type"].value_counts().reindex(CHANGE_LABELS).to_frame("건수").assign(**{"비율(%)": lambda d: (d["건수"] / d["건수"].sum() * 100).round(2)}).to_string())

                                            form → input  corrected_form → target
정규화로 값이 바뀐 행 (공백 제거 또는 NFC)         12289                    12112
그중 NFC 때문에 바뀐 행                                8                        8

[변경 유형 - 빈 행 제외, 전체 데이터]
                    건수  비율(%)
change_type                      
unchanged         154271    13.95
spacing_only       96471     8.72
punct_only        587687    53.13
spelling_or_word  267663    24.20


## 3. 제외 대상 표시 (삭제하지 않음)

다음 행은 **학습·검증·시험 모두에서 제외**하되, 삭제하지 않고 제외 사유와 함께 `excluded_rows.jsonl`에 저장합니다.

- `EMPTY_INPUT` / `EMPTY_TARGET`: `input` 또는 `target`이 빈 문자열
- `HATE_ONLY_INPUT` / `HATE_ONLY_TARGET`: 문장 **전체**가 `hate-speech` 자리표시자(공백·문장부호 제외)
- `HATE_PARTIAL_INPUT` / `HATE_PARTIAL_TARGET`: 정상 문장 **일부**에만 `hate-speech`가 들어 있음
- `LEN_GROWTH_EXTREME` / `DEID_TOKEN_CHANGED`: 정답에 결함이 있는 것으로 확인된 두 유형 (아래 3-0 참고)
- `EMOJI_ONLY_INPUT` / `EMOJI_ONLY_TARGET`: 글자·숫자 없이 이모지(그림 기호)만 있음. 공백·문장부호가 같이 있어도 해당. `ㅋㅋ`, `ㅠㅠ`, `^^` 같은 자모·글자 이모티콘은 해당하지 않음

`hate-speech`는 원래의 유해 발화가 치환된 자리표시자로 보이는 표기입니다. ONLY와 PARTIAL을 사유로 나눠 두었으므로, 나중에 PARTIAL 행을 다시 살릴지 판단할 때 제외 규칙 한 곳만 바꾸면 됩니다(현재는 둘 다 제외). 이모지만 있는 행은 교정할 내용이 없거나(input==target) 이모지가 지워지거나 바뀐 것(input≠target)이라 학습·평가 신호가 약하다고 판단해 제외합니다. 이 판단의 근거는 셀 출력의 건수와 예시입니다. 모든 결정은 행 내용만으로 정해지므로 분할 전에 해도 누수가 아닙니다.

참고: 제외된 행은 시험 세트에서도 빠지므로, 최종 평가 결과는 "자리표시자·이모지만 있는 발화, 빈 발화, 정답이 깨진 행을 뺀 데이터"에 대한 성능입니다. 보고서에 이 점을 적어야 합니다.


### 3-0. 정답에 결함이 있는 두 유형을 제외 사유에 추가

아래 두 가지는 정답 자체가 깨져 있는 경우가 많아 **플래그가 아니라 제외 사유로 올렸습니다**. 근거는 사람이 직접 본 결과입니다(판정자 1명, 눈으로 본 판정이라 추정치).

- `LEN_GROWTH_EXTREME`: 정답이 입력보다 지나치게 **길어진** 경우(공백·문장부호 제외 글자 수 비율 1.5 초과 & 3자 이상 차이). 입력에 없던 문장이 덧붙거나 `ERROR:#REF!` 같은 깨진 표기가 들어간 행이 많음. 제외 대상 101건을 전수 확인했을 때 절반 이상이 결함이었음
- `DEID_TOKEN_CHANGED`: 입력에 없던 이름이 정답에서 `name5`로 바뀌거나 `name2`가 `Name 2`로 깨짐 (train 표본 30개 중 18개가 결함)

**정답이 크게 짧아진 경우(비율 0.7 미만)는 제외하지 않습니다.** `뭐어어어엉엌!!!!!→뭐!`처럼 늘여 쓴 글자를 줄이는 정상 교정이 대부분이라(표본 25개 중 결함 1개), 제외하면 이런 입력이 학습·평가에서 빠져 결과가 실제보다 쉬워집니다. 이 경우는 정보용 플래그 `flag_len_shrink`로만 표시합니다.

두 규칙 모두 **행 내용만으로 결정**되고 train/validation/test에 똑같이 적용합니다. 규칙은 v2 분할의 train 표본을 보고 정했고 test 행 자체는 조정 대상으로 삼지 않았습니다. 분할 전에 제외하므로 train 통계(입력별 정답 종류, 단어 규칙)에도 결함 행이 섞이지 않습니다. 이 제외로 validation/test 평가는 "정답이 깨진 행을 뺀 데이터"에 대한 결과가 됩니다.


In [5]:
# 3-0. 정답에 결함이 있는 것으로 확인된 두 유형 (행 내용만으로 판정, 다른 행의 통계를 쓰지 않음)
valid_rows = df["input"].ne("") & df["target"].ne("")

# (1) 정답이 지나치게 길어짐: 공백·문장부호 제외 글자 수 비율이 1.5 초과이면서 3자 이상 차이 (정답에 내용이 덧붙음). 짧아진 경우는 제외하지 않음
_ratio = df["content_len_tg"] / df["content_len_in"].where(df["content_len_in"] > 0)
_diff = df["content_len_tg"] - df["content_len_in"]
df["ex_len_growth"] = valid_rows & (_ratio > LEN_RATIO_HI) & (_diff >= LEN_ABS_MIN)

# (2) 비식별화 토큰 변경: name1, address 등의 종류·개수가 입력과 정답에서 다름 (대소문자만 다른 경우는 제외하지 않음-정보용 플래그로 남김)
def deid_changed(a, b):
    fa, fb = DEID_RE.findall(a), DEID_RE.findall(b)
    if Counter(fa) == Counter(fb):
        return False
    return Counter(map(str.lower, fa)) != Counter(map(str.lower, fb))

_cand = (df["input"].str.contains(DEID_RE) | df["target"].str.contains(DEID_RE)).to_numpy() & valid_rows.to_numpy()
_a, _b = df["input"].to_numpy(), df["target"].to_numpy()
_res = np.zeros(len(df), dtype=bool)
for _i in np.flatnonzero(_cand):
    _res[_i] = deid_changed(_a[_i], _b[_i])
df["ex_deid_changed"] = _res
del _ratio, _diff, _cand, _a, _b, _res

print("[정답 결함 유형] 정답이 지나치게 길어짐:", f"{int(df['ex_len_growth'].sum()):,}건", "| 비식별화 토큰 변경:", f"{int(df['ex_deid_changed'].sum()):,}건")
print("무작위 예시 (seed=42):")
for _col in ("ex_len_growth", "ex_deid_changed"):
    _s = df[df[_col]]
    print(f"- {_col}")
    for _, _r in _s.sample(min(5, len(_s)), random_state=SEED).iterrows():
        print("   ", repr(_r["input"][:60]), "->", repr(_r["target"][:60]))

[정답 결함 유형] 정답이 지나치게 길어짐: 101건 | 비식별화 토큰 변경: 40건
무작위 예시 (seed=42):
- ex_len_growth
    '아니면 스텐팬' -> '아니면 스테인리스강 팬.'
    '잼써요!' -> '잼써요! ㅋㅋㅋ'
    '깅강씨는요?' -> 'name5 씨는요?'
    '앜ㅋㅋㅋㅋㅋㅋㅋ' -> '아. ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ'
    '헉' -> '헉... 2주에 한 번씩 등교하는군요.'
- ex_deid_changed
    '저는 name5 name6 해쭈 name8 봅니더@!' -> '저는 name5, name6, name7, name8 봅니다! @'
    'zzzzzaccount 누구랑했어' -> 'zzzzz account 누구랑 했어?'
    'name63권샀는데 1.2.10' -> 'name6 3권 샀는데 1.2.10'
    'ㅠㅠname10도 나갔는데 name8까지아프다니ㅠㅠㅠㅠ진짜 넘 소중하고 작은 생명들이라 병걸릴때마다 속탈것같' -> 'ㅠㅠ name10도 나갔는데 name8까지 아프다니. ㅠㅠㅠㅠ 진짜 너name8 소중하고 작은 생명들이라 병'
    '우체국->name210000\r\n우체국->name1242060\r\n잔액830997' -> '우체국 -> ame210000 우체국 -> name1242060 잔액 830997'


In [6]:
df["ex_empty_input"] = df["input"].eq("")
df["ex_empty_target"] = df["target"].eq("")
df["ex_hate_input"] = df["input"].str.contains(HATE_RE)     # hate-speech 포함 여부(전체/일부 구분 전)
df["ex_hate_target"] = df["target"].str.contains(HATE_RE)

def placeholder_only(s: pd.Series) -> pd.Series:
    """자리표시자를 지우고 공백·문장부호도 지웠을 때 아무것도 안 남으면 '문장 전체가 자리표시자'."""
    rest = s.str.replace(HATE_RE, "", regex=True).str.replace(PUNCT_STRIP_RE, "", regex=True)
    return rest.eq("")

# hate-speech 행을 '전체가 자리표시자(ONLY)'와 '일부만 자리표시자(PART)'로 나눠 사유를 따로 기록
for side in ("input", "target"):
    m = df[f"ex_hate_{side}"]
    m_arr = m.to_numpy(dtype=bool)
    only = np.zeros(len(df), dtype=bool)
    only[m_arr] = placeholder_only(df.loc[m, side]).to_numpy(dtype=bool)
    df[f"ex_hate_only_{side}"] = only
    df[f"ex_hate_part_{side}"] = m_arr & ~only

# 이모지만 있는 행 (input 또는 target)
df["ex_emoji_input"] = df["input"].str.fullmatch(EMOJI_ONLY_RE)
df["ex_emoji_target"] = df["target"].str.fullmatch(EMOJI_ONLY_RE)

EX_COLS = {"ex_empty_input": "EMPTY_INPUT", "ex_empty_target": "EMPTY_TARGET",
           "ex_hate_only_input": "HATE_ONLY_INPUT", "ex_hate_only_target": "HATE_ONLY_TARGET",
           "ex_hate_part_input": "HATE_PARTIAL_INPUT", "ex_hate_part_target": "HATE_PARTIAL_TARGET",
           "ex_emoji_input": "EMOJI_ONLY_INPUT", "ex_emoji_target": "EMOJI_ONLY_TARGET",
           "ex_len_growth": "LEN_GROWTH_EXTREME", "ex_deid_changed": "DEID_TOKEN_CHANGED"}
df["excluded"] = df[list(EX_COLS)].any(axis=1)

hate_any = df["ex_hate_input"] | df["ex_hate_target"]
sub = df.loc[hate_any]
in_full, tg_full = placeholder_only(sub["input"]), placeholder_only(sub["target"])
scope = np.select(
    [sub["ex_hate_input"] & sub["ex_hate_target"] & in_full & tg_full,
     sub["ex_hate_input"] & ~sub["ex_hate_target"],
     ~sub["ex_hate_input"] & sub["ex_hate_target"],
     ~(in_full & tg_full)],
    ["input·target 모두 문장 전체가 자리표시자", "input에만 포함", "target에만 포함(input은 일반 문장)", "일부 문장에 포함(양쪽 또는 한쪽)"],
    default="기타")
print("[hate-speech 포함 행]", f"{len(sub):,}건 (전체 발화의 {len(sub)/len(df)*100:.2f}%)")
print(pd.Series(scope).value_counts().to_frame("건수").to_string())
strict = int(((df["input"].str.contains("hate-speech", regex=False)) | (df["target"].str.contains("hate-speech", regex=False))).sum())
print(f"\n정확히 'hate-speech'(소문자, 하이픈)가 든 행: {strict:,} / 표기 변형까지 포함한 정규식: {int(hate_any.sum()):,}")

print("\n[제외 사유별 건수 - 사유가 겹치는 행은 각 사유에 모두 셈]")
ex_table = pd.DataFrame({"건수": {v: int(df[k].sum()) for k, v in EX_COLS.items()}})
ex_table.loc["제외 행 합계(중복 제거)"] = int(df["excluded"].sum())
ex_table["전체 발화 대비 %"] = (ex_table["건수"] / len(df) * 100).round(3)
print(ex_table.to_string())
print(f"\n사용 가능(제외되지 않은) 행: {int((~df['excluded']).sum()):,}")


[hate-speech 포함 행] 9,547건 (전체 발화의 0.85%)
                                          건수
일부 문장에 포함(양쪽 또는 한쪽)          5688
input·target 모두 문장 전체가 자리표시자  3829
target에만 포함(input은 일반 문장)          30

정확히 'hate-speech'(소문자, 하이픈)가 든 행: 9,547 / 표기 변형까지 포함한 정규식: 9,547

[제외 사유별 건수 - 사유가 겹치는 행은 각 사유에 모두 셈]
                          건수  전체 발화 대비 %
EMPTY_INPUT              22980             2.035
EMPTY_TARGET             23271             2.061
HATE_ONLY_INPUT           3829             0.339
HATE_ONLY_TARGET          3861             0.342
HATE_PARTIAL_INPUT        5688             0.504
HATE_PARTIAL_TARGET       5686             0.503
EMOJI_ONLY_INPUT             2             0.000
EMOJI_ONLY_TARGET            2             0.000
LEN_GROWTH_EXTREME         101             0.009
DEID_TOKEN_CHANGED          40             0.004
제외 행 합계(중복 제거)  32951             2.918

사용 가능(제외되지 않은) 행: 1,096,412


In [7]:
# 이모지 판정이 과하게 넓지 않은지 눈으로 확인: 판정된 행의 문자 종류와 무작위 예시
emoji_rows = df[df["ex_emoji_input"] | df["ex_emoji_target"]]
print(f"\n[이모지만 있는 행] {len(emoji_rows):,}건 | input==target: {int((emoji_rows['input'] == emoji_rows['target']).sum()):,}건 | input≠target: {int((emoji_rows['input'] != emoji_rows['target']).sum()):,}건")
print("input≠target 이면서 이모지 판정된 행은 이모지가 지워지거나 다른 글자로 바뀐 경우이므로 라벨 오류 가능성이 높아 함께 제외합니다.")
if len(emoji_rows):
    print("무작위 예시 (seed=42):")
    print(emoji_rows.sample(min(10, len(emoji_rows)), random_state=SEED)[["utterance_id", "input", "target"]].to_string(index=False))
    ch = Counter(c for t in pd.concat([emoji_rows["input"], emoji_rows["target"]]) for c in t if re.fullmatch(EMOJI_BASE, c))
    print("판정에 쓰인 이모지 상위 15개:", ch.most_common(15))



[이모지만 있는 행] 2건 | input==target: 2건 | input≠target: 0건
input≠target 이면서 이모지 판정된 행은 이모지가 지워지거나 다른 글자로 바뀐 경우이므로 라벨 오류 가능성이 높아 함께 제외합니다.
무작위 예시 (seed=42):
        utterance_id input target
MMRW2100000019.11.47    🪨     🪨
MDRW2100001581.1.544    🤎     🤎
판정에 쓰인 이모지 상위 15개: [('🤎', 2), ('🪨', 2)]


In [8]:
print("\n[무작위 예시: target에만 hate-speech가 있고 input은 일반 문장인 경우 (seed=42)]")
ex_only_tg = df[df["ex_hate_target"] & ~df["ex_hate_input"]]
if len(ex_only_tg):
    print(ex_only_tg.sample(min(5, len(ex_only_tg)), random_state=SEED)[["utterance_id", "input", "target"]].to_string(index=False))
print("\n[무작위 예시: 일부만 hate-speech인 행 (HATE_PARTIAL, seed=42) - 남길지 팀에서 판단]")
part = df[df["ex_hate_part_input"] | df["ex_hate_part_target"]]
if len(part):
    print(part.sample(min(10, len(part)), random_state=SEED)[["utterance_id", "input", "target"]].to_string(index=False))



[무작위 예시: target에만 hate-speech가 있고 input은 일반 문장인 경우 (seed=42)]
       utterance_id                                                                                input                   target
MDRW2100037751.1.30                                                                          ㅡㅡ 화나요              hate-speech
MDRW2100037751.1.10                             우선은 좀 더 지켜보기오 했대요 근데 그 왕따당하는 이유가              hate-speech
MDRW2100037751.1.26 맞아요 저 뉴스봤는데 대박인게 우리나라는 부동산 대출제한있어서 한국인들이 못사자나요              hate-speech
MDRW2100037751.1.12                                                                     아미친..........              hate-speech
MDRW2100031402.1.32                                                                  와 미.틴.놈들이네여 와, hate-speech들이네요.

[무작위 예시: 일부만 hate-speech인 행 (HATE_PARTIAL, seed=42) - 남길지 팀에서 판단]
         utterance_id                                       input                                         target
 MDRW2100001156.35.12                        안하는줄 hate-speec

## 4. 문서 단위 90:5:5 분할

같은 `document_id`의 발화는 **한 split에만** 들어갑니다. EDA에서 확인했듯 문서 하나에 26개 이상의 발화가 있고(중앙값 32), 같은 대화의 화자 말투·주제가 반복되므로 발화 단위 분할은 평가를 부풀립니다.

**분할에 쓰는 정보는 메타데이터(플랫폼)와 문서 크기, 입력 문장 텍스트뿐**이며, 정답(`target`)이나 교정 통계는 쓰지 않습니다. 그래서 이 단계는 품질 판단과 독립입니다.

1. **문서 묶음**: 15자 이상·한글 5음절 이상인 같은 입력 문장을 공유하는 문서는 같은 묶음으로 묶어 함께 배정합니다(EDA 6.2: 이런 문서는 소수). `ㅋㅋㅋ…` 같은 자모·부호 위주의 반복은 대화 복제의 증거가 아니라 묶지 않습니다. 끄려면 `GROUP_NEAR_DUP_DOCS = False`.
2. **층화**: 플랫폼 x 문서 크기(3분위) 층마다 목표 비율(0.90/0.05/0.05)에 가장 가깝게 배정합니다. 배정 순서는 고정 시드로 섞습니다.
3. 비율 기준은 **제외되지 않은 행 수**입니다. 최종 사용 가능 행이 90:5:5에 가깝게 나옵니다.

In [9]:
usable = ~df["excluded"]
doc_tbl = df.groupby("document_id", sort=True).agg(publisher=("publisher", "first"), n_rows=("utterance_id", "size"),
                                                    n_usable=("excluded", lambda s: int((~s).sum())))
print(f"문서 {len(doc_tbl):,}개, 문서당 발화 수 중앙값 {doc_tbl['n_rows'].median():.0f}, 최소 {doc_tbl['n_rows'].min()}")

# --- 4-a. 같은 긴 문장을 공유하는 문서 묶기 (입력 문장만 사용) ---
parent = {d: d for d in doc_tbl.index}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

n_linked_forms = 0
if GROUP_NEAR_DUP_DOCS:
    cand = df.loc[usable, ["document_id", "input"]].copy()
    cand["key"] = cand["input"].str.replace(r"\s+", "", regex=True)
    cand = cand[(cand["key"].str.len() >= DUP_MIN_LEN) & (cand["key"].str.count(r"[가-힣]") >= DUP_MIN_HANGUL)]
    cand = cand.drop_duplicates(["document_id", "key"])
    docs_per_key = cand.groupby("key")["document_id"].nunique()
    shared = cand[cand["key"].isin(docs_per_key[docs_per_key >= 2].index)]
    for _, docs in shared.groupby("key")["document_id"]:
        docs = list(docs)
        for other in docs[1:]:
            union(docs[0], other)
    n_linked_forms = int((docs_per_key >= 2).sum())

doc_tbl["group_id"] = [find(d) for d in doc_tbl.index]
grp_sizes = doc_tbl.groupby("group_id").size()
print(f"공유 문장 {n_linked_forms:,}개로 연결된 문서 묶음: {int((grp_sizes >= 2).sum())}개 (묶인 문서 {int(grp_sizes[grp_sizes >= 2].sum())}개, 최대 묶음 {int(grp_sizes.max())}개 문서)")

# --- 4-b. 묶음(=배정 단위)별 층화 배정 ---
units = doc_tbl.groupby("group_id").agg(n_usable=("n_usable", "sum"), n_docs=("n_rows", "size"))
big = doc_tbl.sort_values("n_usable", ascending=False).drop_duplicates("group_id").set_index("group_id")["publisher"]   # 묶음의 대표 플랫폼
units["publisher"] = big.reindex(units.index)
units["size_bin"] = pd.qcut(units["n_usable"].rank(method="first"), 3, labels=["small", "mid", "large"])
units["stratum"] = units["publisher"].astype(str) + "|" + units["size_bin"].astype(str)

rng = np.random.default_rng(SEED)
unit_split = {}
for stratum in sorted(units["stratum"].unique()):
    g = units[units["stratum"] == stratum]
    order = rng.permutation(len(g))
    ids, rows_ = g.index.to_numpy()[order], g["n_usable"].to_numpy()[order]
    cnt, cum = {s: 0 for s in SPLITS}, 0
    for uid, n in zip(ids, rows_):
        cum += n
        deficit = {s: SPLIT_RATIOS[s] * cum - cnt[s] for s in SPLITS}     # 목표 비율 대비 가장 모자란 split에 배정
        s = max(SPLITS, key=lambda k: deficit[k])
        unit_split[uid] = s
        cnt[s] += n

doc_tbl["split"] = doc_tbl["group_id"].map(unit_split)
df["split"] = df["document_id"].map(doc_tbl["split"])

# --- 4-c. 분할 검증 ---
assert df["split"].notna().all()
doc_splits = df.groupby("document_id")["split"].nunique()
assert (doc_splits == 1).all(), "같은 document_id가 여러 split에 걸쳐 있습니다."
grp_splits = doc_tbl.groupby("group_id")["split"].nunique()
assert (grp_splits == 1).all(), "같은 문서 묶음이 여러 split에 걸쳐 있습니다."
print("검증 통과: 문서 단위·묶음 단위로 split이 섞이지 않음")

split_tbl = pd.DataFrame({
    "문서 수": doc_tbl.groupby("split").size(),
    "전체 발화": df.groupby("split").size(),
    "제외 발화": df[df["excluded"]].groupby("split").size(),
    "사용 가능 발화": df[usable].groupby("split").size(),
}).reindex(SPLITS).fillna(0).astype(int)
split_tbl["사용 가능 비율(%)"] = (split_tbl["사용 가능 발화"] / split_tbl["사용 가능 발화"].sum() * 100).round(2)
print(split_tbl.to_string())
pub_share = pd.crosstab(df.loc[usable, "split"], df.loc[usable, "publisher"], normalize="index").reindex(SPLITS).mul(100).round(2)
print("\n[split별 플랫폼 비율(%) - 사용 가능 행 기준]")
print(pub_share.to_string())

문서 21,335개, 문서당 발화 수 중앙값 32, 최소 26
공유 문장 142개로 연결된 문서 묶음: 112개 (묶인 문서 271개, 최대 묶음 10개 문서)
검증 통과: 문서 단위·묶음 단위로 split이 섞이지 않음
            문서 수  전체 발화  제외 발화  사용 가능 발화  사용 가능 비율(%)
split                                                                       
train         18334    1016353      29625          986728              90.00
validation     1532      56386       1656           54730               4.99
test           1469      56624       1670           54954               5.01

[split별 플랫폼 비율(%) - 사용 가능 행 기준]
publisher   심심이  카카오톡
split                       
train        50.63     49.37
validation   50.67     49.33
test         50.53     49.47


## 5. train 데이터에서만 만드는 품질 판단 통계

여기서 만드는 통계는 **오직 train의 사용 가능 행**에서 계산합니다. validation/test 행은 이 표를 *조회*만 하고 통계에 기여하지 않으므로, 시험 정보가 train 쪽 판단에 들어가지 않습니다.

1. **입력별 정답 목록**: train에서 같은 `input`에 어떤 `target`들이 몇 번씩 붙는지. → "동일 form에 여러 corrected_form" 플래그의 근거
2. **단어 교정 규칙**: train에서 띄어쓰기가 바뀌지 않은 행(입력과 정답의 단어 수가 같은 행)의 같은 위치 단어를 비교해 `틀린 단어 → 고친 단어`를 모으고, **거의 항상 고쳐지는 단어(기본: 20회 이상, 90% 이상)** 만 남김. → "교정 누락/부분 교정 의심" 플래그의 근거. 규칙은 문맥을 보지 않는 단순 사전이라 의심 표시용입니다.
3. **참고용 분위수**: 길이 비율·편집량의 train 분위수. 고정 임계값이 train 분포에서 어디쯤인지 눈으로 확인하기 위한 것이며 플래그 계산에는 쓰이지 않습니다.

In [10]:
train = df[usable & df["split"].eq("train")]
print(f"train 사용 가능 행: {len(train):,}")

# --- 5-1. 입력별 정답 목록 ---
pair_cnt = train.groupby(["input", "target"], sort=False).size().rename("n").reset_index()
n_variants = pair_cnt.groupby("input", sort=False)["target"].size()
majority = pair_cnt.sort_values("n", ascending=False).drop_duplicates("input").set_index("input")["target"]
train_conflict_inputs = n_variants[n_variants >= 2]
print(f"train 고유 입력 {len(n_variants):,}개 중 정답이 2종 이상인 입력 {len(train_conflict_inputs):,}개")

# --- 5-2. 단어 교정 규칙 (train에서만) ---
def word_tokens(s):
    return [t for t in (w.translate(PUNCT_TABLE) for w in s.split()) if t]

tok_rows, repl = Counter(), defaultdict(Counter)
t0 = time.time()
for inp, tgt in zip(train["input"].to_numpy(), train["target"].to_numpy()):
    a = word_tokens(inp)
    tok_rows.update(set(a))
    if inp == tgt:
        continue
    b = word_tokens(tgt)
    if len(a) != len(b):
        continue
    for x, y in zip(a, b):
        if x != y:
            repl[x][y] += 1

WORD_RULES = {}
for src, c in repl.items():
    n_repl = sum(c.values())
    dst, top_n = c.most_common(1)[0]
    rate = min(1.0, n_repl / max(1, tok_rows[src]))
    if n_repl >= RULE_MIN_SUPPORT and rate >= RULE_MIN_RATE and top_n / n_repl >= RULE_MIN_RATE:
        WORD_RULES[src] = {"dst": dst, "n_replaced": n_repl, "n_rows_with_src": int(tok_rows[src]), "rate": round(rate, 3)}
print(f"단어 교정 규칙 {len(WORD_RULES):,}개 생성 ({time.time()-t0:.0f}s)")
if WORD_RULES:
    rules_view = pd.DataFrame.from_dict(WORD_RULES, orient="index").sort_values("n_replaced", ascending=False)
    print(rules_view.head(15).to_string())
else:
    print("(생성된 규칙이 없습니다. 점검용 소규모 실행이면 정상입니다.)")
print("※ 규칙은 train의 통계일 뿐이며 문맥을 보지 않습니다. 위 목록을 눈으로 훑어 이상한 규칙이 있는지 확인하세요.")

# --- 5-3. 참고용 분위수 (플래그 계산에는 사용하지 않음) ---
tv = train[train["content_len_in"] > 0]
train_len_ratio = (tv["content_len_tg"] / tv["content_len_in"])
print("\n[train 길이 비율(공백·문장부호 제외) 분위수 - 참고]  고정 임계값:", LEN_RATIO_LO, "/", LEN_RATIO_HI)
print(train_len_ratio.quantile([0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).round(3).to_frame("비율").T.to_string())

train 사용 가능 행: 986,728
train 고유 입력 828,984개 중 정답이 2종 이상인 입력 8,452개
단어 교정 규칙 24개 생성 (4s)
                       dst  n_replaced  n_rows_with_src   rate
ㅇㅋ                  오키         452              488  0.926
ㅇㅈ                  인정         428              475  0.901
ㅇㅇㅇㅇ          응응응응         255              277  0.921
ㅇㅋㅇㅋ          오키오키         239              264  0.905
ㅇㅇㅇㅇㅇ      응응응응응         104              114  0.912
알겟어              알겠어          90               99  0.909
시상에              세상에          70               72  0.972
응ㅇ응응          응응응응          54               60  0.900
알았어용          알았어요          50               54  0.926
알겟엉              알겠어          46               47  0.979
안뇽                  안녕          42               45  0.933
ㅇㅇㅇㅇㅇㅇ  응응응응응응          39               40  0.975
안녕하세여      안녕하세요          37               40  0.925
먹엇어              먹었어          33               36  0.917
마즘                  맞음          31               34  0.912
※ 규칙은 tr

## 6. 품질 플래그 부여 (자동 삭제 없음)

아래 규칙을 **train/validation/test에 똑같이** 적용해 `flag_*` 컬럼을 만듭니다. 플래그는 "의심"이지 "정답 오류 확정"이 아닙니다. 제외 대상(빈 행, `hate-speech`, 이모지만 있는 행, 정답이 지나치게 길어진 행, 비식별화 토큰 변경)은 이미 표시했으므로 플래그는 사용 가능한 행에만 붙입니다. 정답이 지나치게 길어진 경우와 비식별화 토큰 변경은 3-0에서 제외 사유가 되었으므로 더 이상 플래그로 만들지 않습니다.

| 플래그 | 무엇을 잡는가 | 통계의 출처 | 정상인데 걸릴 수 있는 경우 |
|---|---|---|---|
| `flag_form_conflict`, `flag_form_conflict_long` | train에서 같은 입력에 정답이 2종 이상 붙음 (`_long`: 공백·문장부호 제외 5자 이상) | train 표 조회 | 문맥이 다른 동형어, 아주 짧은 발화 |
| `flag_edit_large` | 철자·단어 수준 변경 중 편집량 0.5 이상 | 고정 임계값 | 짧은 발화의 어절 교체 |
| `flag_digit_changed`, `flag_alpha_changed` | 숫자열/영문열이 달라짐(영문의 대소문자만 변경은 제외) | 규칙 | 숫자의 한글 표기 변환 |
| `flag_emoji_changed` | 이모지·문자 이모티콘(`^^`, `-_-`)이 달라짐 | 규칙 | 드묾 |
| `flag_punct_excess` | 문장부호가 3개 이상 늘었거나 같은 부호 4개 이상 연속이 새로 생김 | 고정 임계값 | 인용부호·쉼표가 많은 문장 |
| `flag_missed_correction_suspect` | 입력이 그대로인데, train에서 거의 항상 고쳐지는 단어가 들어 있음 | train 단어 규칙 | 문맥상 정상인 단어 |
| `flag_partial_correction_suspect` | 일부는 고쳐졌지만, train에서 거의 항상 고쳐지는 단어가 정답에 그대로 남음 | train 단어 규칙 | 위와 같음 |

**정보용 플래그**(품질 점수에는 넣지 않음): `flag_len_shrink`(공백·문장부호 제외 길이 비율이 0.7 미만이고 3자 이상 짧아짐. 늘여 쓴 글자를 줄이는 정상 교정이 대부분이라 제외하지 않음), `flag_deid_case_only`(`name1`→`Name1`처럼 대소문자만 변경), `flag_input_has_linebreak`(입력 안쪽에 줄바꿈·탭), `flag_seen_input_in_train` / `flag_seen_pair_in_train`(validation/test 행의 입력 또는 입력·정답 쌍이 train에 이미 있음 → 평가 점수가 쉬운 반복 표현으로 부풀 수 있는지 확인용).

**한계**: 고정 임계값(0.7/1.5/3자, 편집량 0.5, 부호 3개)은 EDA에서 정한 초기값이라 train 라벨 통계로 학습한 값이 아닙니다. validation/test의 `flag_form_conflict*`와 단어 규칙 플래그는 **train에 있는 입력·단어만** 잡으므로 train보다 적게 나오는 것이 정상입니다(하한값).

In [11]:
U = usable.to_numpy()
inp, tgt = df["input"], df["target"]

# --- 6-1. train 표를 조회하는 플래그 ---
df["train_n_variants"] = inp.map(n_variants).fillna(0).astype(int)
df["flag_form_conflict"] = U & (df["train_n_variants"] >= 2)
df["flag_form_conflict_long"] = df["flag_form_conflict"] & (df["content_len_in"] >= CONFLICT_MIN_LEN)

# --- 6-2. 길이·편집량·문장부호 ---
ratio = (df["content_len_tg"] / df["content_len_in"].where(df["content_len_in"] > 0))
diff_len = df["content_len_tg"] - df["content_len_in"]
df["len_ratio"] = ratio.where(U)
df["flag_len_shrink"] = U & (ratio < LEN_RATIO_LO) & (-diff_len >= LEN_ABS_MIN)   # 정보용: 정답이 크게 짧아짐(늘여 쓴 글자 줄이기 등 정상 교정이 대부분)

df["edit_ratio"] = np.nan
idx3 = np.flatnonzero(U & (df["change_type"] == "spelling_or_word").to_numpy())
a_arr, b_arr = inp.to_numpy(), tgt.to_numpy()
er = np.array([1.0 - difflib.SequenceMatcher(None, a_arr[i], b_arr[i], autojunk=False).ratio() for i in idx3])
df.loc[df.index[idx3], "edit_ratio"] = er
df["flag_edit_large"] = U & df["edit_ratio"].ge(EDIT_RATIO_HI)

df["punct_added"] = (tgt.str.count(PUNCT_RE) - inp.str.count(PUNCT_RE)).where(U)
REPEAT_RE = re.compile(r"([.!?~…])\1{%d,}" % (PUNCT_RUN_MIN - 1))
new_repeat = tgt.str.contains(REPEAT_RE) & ~inp.str.contains(REPEAT_RE)
df["flag_punct_excess"] = U & ((df["punct_added"] >= PUNCT_ADD_MIN) | new_repeat)

# --- 6-3. 숫자·영문·비식별화 토큰·이모티콘 변화 ---
DIGIT_RE, ALPHA_RE = re.compile(r"\d+"), re.compile(r"[A-Za-z]+")
EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍]")
TEXTEMO_RE = re.compile(r"\^_*\^|T_T|-_-|;;+|:-?[)(D]")

def _strip_deid(s):
    return DEID_RE.sub(" ", s)

def multiset_kind(a, b, soft=None):
    ca, cb = Counter(a), Counter(b)
    if ca == cb:
        return ""
    lost, added = ca - cb, cb - ca
    if lost and not added:
        return "lost"
    if added and not lost:
        return "added"
    if soft and Counter(map(soft, a)) == Counter(map(soft, b)):
        return "case_only"
    return "replaced"

def family_change(rx, pre=None, soft=None):
    cand = (inp.str.contains(rx) | tgt.str.contains(rx)).to_numpy() & U
    res = np.full(len(df), "", dtype=object)
    for i in np.flatnonzero(cand):
        a, b = a_arr[i], b_arr[i]
        if pre:
            a, b = pre(a), pre(b)
        res[i] = multiset_kind(rx.findall(a), rx.findall(b), soft)
    return res

t0 = time.time()
deid_kind = family_change(DEID_RE, soft=str.lower)
digit_kind = family_change(DIGIT_RE, pre=_strip_deid)
alpha_kind = family_change(ALPHA_RE, pre=_strip_deid, soft=str.lower)
emoji_kind = family_change(EMOJI_RE)
textemo_kind = family_change(TEXTEMO_RE)
df["flag_deid_case_only"] = deid_kind == "case_only"
df["flag_digit_changed"] = digit_kind != ""
df["flag_alpha_changed"] = np.isin(alpha_kind, ["lost", "added", "replaced"])
df["flag_emoji_changed"] = (emoji_kind != "") | (textemo_kind != "")
print(f"토큰 비교 {time.time()-t0:.0f}s")

# --- 6-4. train 단어 규칙으로 교정 누락/부분 교정 의심 ---
missed = np.zeros(len(df), dtype=bool)
partial = np.zeros(len(df), dtype=bool)
for i in np.flatnonzero(U):
    hits = [t for t in word_tokens(a_arr[i]) if t in WORD_RULES]
    if not hits:
        continue
    if any(h in set(word_tokens(b_arr[i])) for h in hits):     # 고쳐져야 할 단어가 정답에 그대로 남음
        if a_arr[i] == b_arr[i]:
            missed[i] = True
        else:
            partial[i] = True
df["flag_missed_correction_suspect"] = missed
df["flag_partial_correction_suspect"] = partial

# --- 6-5. 정보용 플래그 ---
df["flag_input_has_linebreak"] = U & inp.str.contains(r"[\r\n\t]").to_numpy()
nontrain = (U & df["split"].ne("train").to_numpy())
df["flag_seen_input_in_train"] = False
df["flag_seen_pair_in_train"] = False
if nontrain.any():
    df.loc[nontrain, "flag_seen_input_in_train"] = df.loc[nontrain, "input"].isin(pd.Index(n_variants.index)).to_numpy()
    train_pairs = pd.MultiIndex.from_frame(pair_cnt[["input", "target"]])
    df.loc[nontrain, "flag_seen_pair_in_train"] = pd.MultiIndex.from_frame(df.loc[nontrain, ["input", "target"]]).isin(train_pairs)

QUALITY_FLAGS = ["flag_form_conflict", "flag_form_conflict_long", "flag_edit_large",
                 "flag_digit_changed", "flag_alpha_changed", "flag_emoji_changed",
                 "flag_punct_excess", "flag_missed_correction_suspect", "flag_partial_correction_suspect"]
INFO_FLAGS = ["flag_len_shrink", "flag_deid_case_only", "flag_input_has_linebreak", "flag_seen_input_in_train", "flag_seen_pair_in_train"]
for c in QUALITY_FLAGS + INFO_FLAGS:
    df[c] = df[c].fillna(False).astype(bool)

M = df[QUALITY_FLAGS + INFO_FLAGS].to_numpy()
names = np.array(QUALITY_FLAGS + INFO_FLAGS)
ql = [list(names[r]) if r.any() else [] for r in M]
df["quality_flags"] = ql                                                       # 정보용 플래그 포함, 표시용 목록
df["n_quality_flags"] = df[QUALITY_FLAGS].sum(axis=1).astype(int)              # 품질 플래그 개수(정보용 제외)
print("플래그 부여 완료")

토큰 비교 5s
플래그 부여 완료


## 7. split별 요약 (제외 수, 플래그 수, 사용 가능 행 수)

표의 비율은 **각 split의 사용 가능 행** 대비입니다. 플래그 수는 "의심 후보 수"이며 오류 수가 아닙니다. 아래 표를 보고 확인할 것:

- 세 split의 변경 유형 분포와 길이 분포가 크게 다르지 않은지 (크게 다르면 분할 방식을 다시 볼 신호)
- validation/test의 플래그 비율이 train보다 낮은 것은 train 표 조회 방식 때문이며 정상입니다(위 한계 참고).

In [12]:
u = df[usable]
FLAG_ALL = QUALITY_FLAGS + INFO_FLAGS
summary = {}
for s in SPLITS:
    d_all, d_u = df[df["split"] == s], u[u["split"] == s]
    summary[s] = {
        "documents": int(d_all["document_id"].nunique()),
        "rows_total": int(len(d_all)),
        "excluded_by_reason": {v: int(d_all[k].sum()) for k, v in EX_COLS.items()},
        "excluded_any": int(d_all["excluded"].sum()),
        "rows_usable": int(len(d_u)),
        "flags": {f: int(d_u[f].sum()) for f in FLAG_ALL},
        "flags_pct_of_usable": {f: round(float(d_u[f].mean() * 100), 3) if len(d_u) else None for f in FLAG_ALL},
        "rows_with_any_quality_flag": int((d_u["n_quality_flags"] > 0).sum()),
        "rows_without_quality_flag": int((d_u["n_quality_flags"] == 0).sum()),
        "change_type_pct": {k: round(float(v), 2) for k, v in (d_u["change_type"].value_counts(normalize=True).reindex(CHANGE_LABELS).fillna(0) * 100).items()},
        "input_len_quantiles": {str(q): float(d_u["input_len"].quantile(q)) for q in (0.5, 0.95, 0.99)},
    }


In [13]:
tbl = pd.DataFrame({
    s: {"문서 수": v["documents"], "전체 행": v["rows_total"], "제외 행(중복 제거)": v["excluded_any"],
        **{f"  제외: {k}": n for k, n in v["excluded_by_reason"].items()},
        "사용 가능 행": v["rows_usable"], "품질 플래그가 1개 이상인 행": v["rows_with_any_quality_flag"], "품질 플래그 없는 행": v["rows_without_quality_flag"]}
    for s, v in summary.items()})
print(tbl.to_string())


                               train  validation   test
문서 수                        18334        1532   1469
전체 행                      1016353       56386  56624
제외 행(중복 제거)             29625        1656   1670
  제외: EMPTY_INPUT            21222         912    846
  제외: EMPTY_TARGET           21488         926    857
  제외: HATE_ONLY_INPUT         2951         401    477
  제외: HATE_ONLY_TARGET        2977         401    483
  제외: HATE_PARTIAL_INPUT      5046         320    322
  제외: HATE_PARTIAL_TARGET     5043         320    323
  제외: EMOJI_ONLY_INPUT           2           0      0
  제외: EMOJI_ONLY_TARGET          2           0      0
  제외: LEN_GROWTH_EXTREME        85           8      8
  제외: DEID_TOKEN_CHANGED        37           2      1
사용 가능 행                  986728       54730  54954
품질 플래그가 1개 이상인 행   153410        7801   7246
품질 플래그 없는 행           833318       46929  47708


In [14]:
flag_tbl = pd.DataFrame({s: {f: f"{v['flags'][f]:,} ({v['flags_pct_of_usable'][f]}%)" for f in FLAG_ALL} for s, v in summary.items()})
print("\n[플래그별 건수 (사용 가능 행 대비 %)]  ※ 위 9개가 품질 플래그, 아래 5개는 정보용")
print(flag_tbl.to_string())



[플래그별 건수 (사용 가능 행 대비 %)]  ※ 위 9개가 품질 플래그, 아래 5개는 정보용
                                            train       validation             test
flag_form_conflict               116,635 (11.82%)  5,550 (10.141%)   4,971 (9.046%)
flag_form_conflict_long           17,095 (1.732%)     868 (1.586%)     870 (1.583%)
flag_edit_large                   23,458 (2.377%)   1,260 (2.302%)   1,160 (2.111%)
flag_digit_changed                 1,372 (0.139%)      74 (0.135%)      86 (0.156%)
flag_alpha_changed                   236 (0.024%)      12 (0.022%)       11 (0.02%)
flag_emoji_changed                   376 (0.038%)      41 (0.075%)      30 (0.055%)
flag_punct_excess                 22,165 (2.246%)   1,278 (2.335%)     1,374 (2.5%)
flag_missed_correction_suspect        23 (0.002%)       1 (0.002%)         0 (0.0%)
flag_partial_correction_suspect        6 (0.001%)       2 (0.004%)         0 (0.0%)
flag_len_shrink                      385 (0.039%)      37 (0.068%)      20 (0.036%)
flag_deid_case_only   

In [15]:
ct = pd.DataFrame({s: v["change_type_pct"] for s, v in summary.items()})
print("\n[변경 유형 분포(%) - 사용 가능 행]")
print(ct.to_string())
lq = pd.DataFrame({s: v["input_len_quantiles"] for s, v in summary.items()})
print("\n[입력 길이 분위수(글자) - 사용 가능 행]")
print(lq.to_string())



[변경 유형 분포(%) - 사용 가능 행]
                  train  validation   test
unchanged         13.86       13.46  12.83
spacing_only       8.75        8.86   8.73
punct_only        53.29       51.85  52.55
spelling_or_word  24.10       25.83  25.89

[입력 길이 분위수(글자) - 사용 가능 행]
      train  validation  test
0.5    11.0        11.0  12.0
0.95   33.0        33.0  34.0
0.99   52.0        53.0  54.0


In [16]:
print("\n[플래그별 무작위 예시 2건 (train, seed=42) - 의심 사례이며 정답 오류로 단정하지 않음]")
tu = u[u["split"] == "train"]
for f in QUALITY_FLAGS:
    sub = tu[tu[f]]
    if len(sub):
        for _, r in sub.sample(min(2, len(sub)), random_state=SEED).iterrows():
            print(f"{f:34s} | {r['input']!r}  ->  {r['target']!r}")



[플래그별 무작위 예시 2건 (train, seed=42) - 의심 사례이며 정답 오류로 단정하지 않음]
flag_form_conflict                 | '안녕하세요'  ->  '안녕하세요?'
flag_form_conflict                 | '웅웅'  ->  '웅웅.'
flag_form_conflict_long            | '써보셨나요?'  ->  '써 보셨나요?'
flag_form_conflict_long            | '아ㅋㅋㅋㅋㅋㅋㅋㅋ'  ->  '아, ㅋㅋㅋㅋㅋㅋㅋㅋ'
flag_edit_large                    | '짱마싯슴!!'  ->  '짱 맛있음!'
flag_edit_large                    | '마자'  ->  '맞아.'
flag_digit_changed                 | '롯데텔로ㅋㅋ26000원씩할인됨'  ->  '롯데텔로 ㅋㅋ 26,000원씩 할인됨.'
flag_digit_changed                 | '그러니까요 아이옷 사야하는ㄷ0'  ->  '그러니까요. 아이 옷 사야 하는데'
flag_alpha_changed                 | '이게무슨일일이야?ㅁㅁ'  ->  '이게 무슨 일이야? Aa'
flag_alpha_changed                 | '아하 ㅎㅎㅎ 저도 한땐 kbl도 챙겨봤는데'  ->  '아하. ㅎㅎㅎ 저도 한땐 케이비엘도 챙겨 봤는데'
flag_emoji_changed                 | '근데 40만원이라니 ㅠㅠ;;'  ->  '근데, 40만 원이라니... ㅠㅠ'
flag_emoji_changed                 | '..........;;;좋다니깐 우선 사보는거 같아여..'  ->  '좋다니깐 우선 사보는 거 같아요...'
flag_punct_excess                  | '와 소고기에 라면이라니 진짜 맛있겠다 와 대박'  ->  

## 8. 저장 (새 폴더, 덮어쓰기 없음)

| 파일 | 내용 |
|---|---|
| `train.jsonl`, `validation.jsonl`, `test.jsonl` | **사용 가능한 행만**. `input`, `target`, `document_id`, `split`, 품질 플래그 컬럼, 원문 컬럼 포함 (원문 컬럼 때문에 train은 약 1GB 이상. 줄이려면 설정의 `INCLUDE_RAW_COLUMNS_IN_SPLIT_FILES = False`) |
| `excluded_rows.jsonl` | 제외한 행 전부와 제외 사유(`exclude_reasons`). 삭제하지 않고 보존 |
| `split_summary.json`, `split_summary.csv` | split별 제외 수·플래그 수·사용 가능 행 수 |
| `document_split_map.csv` | 문서별 split과 묶음(group) 배정 |
| `train_input_conflicts.csv` | train에서 같은 입력에 정답이 여러 개인 목록 |
| `quality_rules_train.json` | train에서 만든 단어 교정 규칙과 임계값 |
| `preprocess_config.json` | 설정·환경·원본 SHA-256(시작/끝) |

플래그가 붙은 행도 학습 파일에 **그대로 들어 있습니다.** 어떤 플래그를 제외 기준으로 쓸지는 이 파일을 읽는 학습 단계에서 정하세요(예: `n_quality_flags == 0`만 사용, 또는 특정 플래그만 제외).

In [17]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_COLS = (["document_id", "utterance_id", "speaker_id", "utterance_index", "split", "publisher", "topic", "relation",
               "speaker_sex", "speaker_age", "input", "target", "change_type", "input_len", "target_len",
               "len_ratio", "edit_ratio", "punct_added", "train_n_variants"]
              + QUALITY_FLAGS + INFO_FLAGS + ["n_quality_flags", "quality_flags"]
              + (["form_raw", "corrected_form_raw", "original_form"] if INCLUDE_RAW_COLUMNS_IN_SPLIT_FILES else []))

def write_jsonl(frame, positions, path, cols, chunk=50_000):
    """메모리를 아끼려고 행 위치(positions)를 chunk 단위로 잘라 저장합니다."""
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for s in range(0, len(positions), chunk):
            text = frame.iloc[positions[s:s + chunk]][cols].to_json(orient="records", lines=True, force_ascii=False)
            f.write(text if text.endswith("\n") else text + "\n")

split_arr = df["split"].to_numpy()
for s in SPLITS:
    write_jsonl(df, np.flatnonzero(usable.to_numpy() & (split_arr == s)), OUT_DIR / f"{s}.jsonl", SPLIT_COLS)

# 제외 행 (삭제하지 않고 사유와 함께 보존)
ex = df[df["excluded"]].copy()
ex["exclude_reasons"] = [[v for k, v in EX_COLS.items() if row[k]] for row in ex[list(EX_COLS)].to_dict("records")]
EX_OUT_COLS = ["document_id", "utterance_id", "speaker_id", "utterance_index", "split", "exclude_reasons", "publisher", "topic",
               "relation", "input", "target", "form_raw", "corrected_form_raw", "original_form"]
write_jsonl(ex, np.arange(len(ex)), OUT_DIR / "excluded_rows.jsonl", EX_OUT_COLS)

# 문서 → split 배정표
doc_tbl.reset_index().rename(columns={"index": "document_id"})[["document_id", "group_id", "split", "publisher", "n_rows", "n_usable"]] \
    .to_csv(OUT_DIR / "document_split_map.csv", index=False, encoding="utf-8-sig")

# train에서 만든 통계
conf_rows = pair_cnt[pair_cnt["input"].isin(train_conflict_inputs.index)].sort_values(["input", "n"], ascending=[True, False])
conf_out = conf_rows.groupby("input", sort=False).agg(n_variants=("target", "size"), total=("n", "sum"),
                                                     targets=("target", lambda s: " | ".join(s)), counts=("n", lambda s: " | ".join(map(str, s)))).reset_index()
conf_out.to_csv(OUT_DIR / "train_input_conflicts.csv", index=False, encoding="utf-8-sig")

THRESHOLDS = {"LEN_RATIO_LO": LEN_RATIO_LO, "LEN_RATIO_HI": LEN_RATIO_HI, "LEN_ABS_MIN": LEN_ABS_MIN, "EDIT_RATIO_HI": EDIT_RATIO_HI,
              "PUNCT_ADD_MIN": PUNCT_ADD_MIN, "PUNCT_RUN_MIN": PUNCT_RUN_MIN, "CONFLICT_MIN_LEN": CONFLICT_MIN_LEN,
              "RULE_MIN_SUPPORT": RULE_MIN_SUPPORT, "RULE_MIN_RATE": RULE_MIN_RATE, "DUP_MIN_LEN": DUP_MIN_LEN, "DUP_MIN_HANGUL": DUP_MIN_HANGUL}
with open(OUT_DIR / "quality_rules_train.json", "w", encoding="utf-8", newline="\n") as f:
    json.dump({"note": "train 사용 가능 행에서만 만든 규칙. validation/test는 조회만 함.", "thresholds_fixed_from_EDA": THRESHOLDS,
               "train_rows": int(len(train)), "train_unique_inputs": int(len(n_variants)), "train_conflict_inputs": int(len(train_conflict_inputs)),
               "word_rules": WORD_RULES}, f, ensure_ascii=False, indent=1)

# 요약
SRC_SHA_END = sha256_of(DATA_PATH)
summary_out = {
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "source": {"path": str(DATA_PATH), "size_bytes": SRC_SIZE, "sha256_start": SRC_SHA_START, "sha256_end": SRC_SHA_END, "unchanged": SRC_SHA_START == SRC_SHA_END},
    "totals": {"documents": int(df["document_id"].nunique()), "rows": int(len(df)), "excluded": int(df["excluded"].sum()), "usable": int(usable.sum())},
    "split_ratios_target": SPLIT_RATIOS, "seed": SEED, "max_docs_smoke_test": MAX_DOCS,
    "document_groups": {"enabled": GROUP_NEAR_DUP_DOCS, "groups_with_2plus_docs": int((grp_sizes >= 2).sum()), "largest_group_docs": int(grp_sizes.max())},
    "quality_flags": QUALITY_FLAGS, "info_flags": INFO_FLAGS,
    "splits": summary,
}
with open(OUT_DIR / "split_summary.json", "w", encoding="utf-8", newline="\n") as f:
    json.dump(summary_out, f, ensure_ascii=False, indent=1)
tbl.T.reset_index().rename(columns={"index": "split"}).to_csv(OUT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
flag_rows = [{"split": s, "flag": f, "count": summary[s]["flags"][f], "pct_of_usable": summary[s]["flags_pct_of_usable"][f]} for s in SPLITS for f in FLAG_ALL]
pd.DataFrame(flag_rows).to_csv(OUT_DIR / "flag_summary.csv", index=False, encoding="utf-8-sig")

with open(OUT_DIR / "preprocess_config.json", "w", encoding="utf-8", newline="\n") as f:
    json.dump({"env": ENV_INFO, "seed": SEED, "split_ratios": SPLIT_RATIOS, "thresholds": THRESHOLDS,
               "hate_regex": HATE_RE.pattern, "emoji_only_regex": EMOJI_ONLY_RE.pattern, "deid_regex": DEID_RE.pattern, "text_normalization": "NFC + strip (앞뒤 공백) only",
               "source_sha256_start": SRC_SHA_START, "source_sha256_end": SRC_SHA_END}, f, ensure_ascii=False, indent=1)

print("저장 완료:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size/1e6:8.1f} MB")
print("원본 SHA-256(끝):", SRC_SHA_END, "| 시작과 동일:", SRC_SHA_START == SRC_SHA_END)

저장 완료: C:\Users\KDS-18\Desktop\pro4_team3\data\preprocessed_v3b
  document_split_map.csv            1.3 MB
  excluded_rows.jsonl              13.4 MB
  flag_summary.csv                  0.0 MB
  preprocess_config.json            0.0 MB
  quality_rules_train.json          0.0 MB
  split_summary.csv                 0.0 MB
  split_summary.json                0.0 MB
  test.jsonl                       62.0 MB
  train.jsonl                    1096.7 MB
  train_input_conflicts.csv         0.5 MB
  validation.jsonl                 61.8 MB
원본 SHA-256(끝): 592e81cf9c41f6ee7e30dc660cb8d3f1c282275deca7eca25d64ab1a9acdec39 | 시작과 동일: True


## 9. 저장 결과 재검증

메모리의 표가 아니라 **저장된 파일을 다시 읽어서** 확인합니다. 하나라도 실패하면 `AssertionError`로 멈춥니다.

In [18]:
# 저장된 파일을 한 줄씩 읽으며 검증합니다 (메모리를 아끼려고 통째로 올리지 않음)
def _content_len(s):
    return len(re.sub(r"\s+", "", s).translate(PUNCT_TABLE))

def iter_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

stat = {}
doc_sets, utt_ids = {}, set()
for s in SPLITS:
    n = n_clean = 0
    docs = set()
    for r in iter_jsonl(OUT_DIR / f"{s}.jsonl"):
        assert r["split"] == s
        assert r["input"] != "" and r["target"] != "", f"{s}: 빈 input/target이 있습니다."
        assert not HATE_RE.search(r["input"]) and not HATE_RE.search(r["target"]), f"{s}: hate-speech 행이 남아 있습니다."
        assert EMOJI_ONLY_RE.fullmatch(r["input"]) is None and EMOJI_ONLY_RE.fullmatch(r["target"]) is None, f"{s}: 이모지만 있는 행이 남아 있습니다."
        _ci, _ct = _content_len(r["input"]), _content_len(r["target"])
        assert not (_ci > 0 and _ct / _ci > LEN_RATIO_HI and _ct - _ci >= LEN_ABS_MIN), f"{s}: 정답이 지나치게 길어진 행이 남아 있습니다."
        assert not deid_changed(r["input"], r["target"]), f"{s}: 비식별화 토큰이 바뀐 행이 남아 있습니다."
        assert unicodedata.is_normalized("NFC", r["input"]) and unicodedata.is_normalized("NFC", r["target"]), f"{s}: NFC가 아닌 텍스트가 있습니다."
        assert r["utterance_id"] not in utt_ids, "발화 ID가 중복됩니다."
        utt_ids.add(r["utterance_id"])
        docs.add(r["document_id"])
        n += 1
        n_clean += (r["n_quality_flags"] == 0)
    stat[s] = {"문서": len(docs), "사용 가능 행": n, "품질 플래그 없는 행": n_clean}
    doc_sets[s] = docs

n_excluded = 0
for r in iter_jsonl(OUT_DIR / "excluded_rows.jsonl"):
    assert len(r["exclude_reasons"]) >= 1
    assert r["utterance_id"] not in utt_ids, "제외 행이 학습 파일에도 들어 있습니다."
    n_excluded += 1

n_usable_files = sum(v["사용 가능 행"] for v in stat.values())
assert n_usable_files + n_excluded == len(df), "사용 가능 + 제외 행 수가 원본 발화 수와 다릅니다."
assert n_usable_files == int(usable.sum())
assert not (doc_sets["train"] & doc_sets["validation"]) and not (doc_sets["train"] & doc_sets["test"]) and not (doc_sets["validation"] & doc_sets["test"]), "split 사이에 같은 document_id가 있습니다."
assert sha256_of(DATA_PATH) == SRC_SHA_START, "원본 파일이 변경되었습니다!"

print("검증 통과")
print(f"- 사용 가능 {n_usable_files:,} + 제외 {n_excluded:,} = 전체 {len(df):,}")
print("- split 간 document_id 겹침 0건, 빈 input/target 0건, hate-speech 잔존 0건, 이모지만 있는 행 잔존 0건, 정답이 지나치게 길어진 행 잔존 0건, 비식별화 토큰 변경 행 잔존 0건, NFC 확인, 발화 ID 중복 0건")
print("- 원본 JSON 해시 변경 없음")
print("\n[최종 split 요약]")
print(pd.DataFrame(stat).T.to_string())

검증 통과
- 사용 가능 1,096,412 + 제외 32,951 = 전체 1,129,363
- split 간 document_id 겹침 0건, 빈 input/target 0건, hate-speech 잔존 0건, 이모지만 있는 행 잔존 0건, 정답이 지나치게 길어진 행 잔존 0건, 비식별화 토큰 변경 행 잔존 0건, NFC 확인, 발화 ID 중복 0건
- 원본 JSON 해시 변경 없음

[최종 split 요약]
             문서  사용 가능 행  품질 플래그 없는 행
train       18328        986728               833318
validation   1527         54730                46929
test         1465         54954                47708


## 10. 이 결과를 쓰는 방법과 한계

**학습 단계에서 정할 것**
- 어떤 플래그로 학습 행을 거를지: 예) `n_quality_flags == 0`만 사용 / `flag_form_conflict_long`, `flag_deid_changed`, `flag_missed_correction_suspect`만 제외. 기준을 바꿀 때는 **validation으로만 비교**하고 test는 마지막에 한 번만 봅니다.
- 플래그로 거른 학습 데이터와 거르지 않은 학습 데이터를 같은 조건에서 비교 실험하면, 이 플래그가 실제로 도움이 되는지 근거를 만들 수 있습니다.

**이 노트북이 하지 않은 것 / 한계**
- 제외한 것은 빈 행, `hate-speech` 포함 행, 이모지만 있는 행, 정답이 깨진 것으로 확인된 두 유형(정답이 지나치게 길어진 행, 비식별화 토큰 변경)뿐입니다. 나머지는 플래그입니다. 플래그가 붙은 행이 진짜 오류인지는 사람이 표본으로 확인하기 전에는 알 수 없습니다.
- 고정 임계값은 EDA(전체 데이터)에서 정한 초기값입니다. train 라벨로 학습한 값이 아닙니다.
- 교정 누락은 정답이 없어 완전히 측정할 수 없습니다. 단어 규칙 플래그는 train에서 거의 항상 고쳐지는 단어에 한정한 **하한 추정**입니다.
- 입력 안쪽의 줄바꿈(`\r\n`)은 NFC + 앞뒤 공백 제거만 적용한다는 조건에 따라 그대로 둡니다. `flag_input_has_linebreak`로 표시만 했습니다.
- `name1` → `Name1`처럼 교정문에서 비식별화 토큰의 대소문자만 바뀐 경우는 `flag_deid_case_only`로 표시만 했고 값은 바꾸지 않았습니다.
- 문서 단위로 나눠도 `안녕하세요`, `ㅋㅋ` 같은 짧은 표현은 split 사이에 겹칩니다. `flag_seen_input_in_train`, `flag_seen_pair_in_train`으로 그 비율을 확인하고, 평가 결과는 이 행을 뺀 값과 함께 보고하는 것을 권합니다.
- 같은 화자가 여러 대화에 나오는지(화자 단위 누수)는 이 노트북에서 확인하지 않았습니다.